# L3-40：结果汇总

读取同一 session 的前四个 JSON。只有 Ascon release、非 Ascon GPIO 和非 Ascon DMA
全部为 `PASS`，且后三者为 `hardware_measured`、`simulated=false`，才给出完整 L3 PASS。

只完成部分阶段时可显式设置 `L3_ALLOW_PARTIAL=1`，结果只能是 `PARTIAL`。

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


repo = Path("/home/xilinx/jupyter_notebooks/amd_fpga_2026").resolve()
assert (repo / "skill/ab_eval_pynq/test_board").is_dir(), (
    "fixed repository path does not exist: %s" % repo
)
run_dir = (
    Path(os.environ["L3_RUN_DIR"]).resolve()  # 可选：不填时读取 l3_current_session.json
    if os.environ.get("L3_RUN_DIR")
    else Path(
        json.loads(
            (repo / "data/skill/l3_current_session.json").read_text(encoding="utf-8")
        )["run_dir"]
    ).resolve()
)

cases = [
    ("00_preflight", "00_preflight.json", "environment"),
    ("10_ascon_release", "10_ascon_release.json", "hardware_measured"),
    ("20_non_ascon_gpio", "20_non_ascon_gpio.json", "hardware_measured"),
    ("30_non_ascon_dma", "30_non_ascon_dma.json", "hardware_measured"),
]
rows = []  # 自动汇总每个阶段的状态，不需要手填
problems = []  # 自动收集所有未通过项
dma_deferred_problems = []  # 仅 DMA 未提供的可延期项
fatal_problems = []  # 其余阶段失败，禁止降级为 PARTIAL
for case, filename, required_claim in cases:
    path = run_dir / filename
    if not path.is_file():
        rows.append({"case": case, "status": "MISSING", "path": str(path)})
        item = "%s is missing" % case
        problems.append(item)
        (dma_deferred_problems if case == "30_non_ascon_dma" else fatal_problems).append(item)
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    row = {
        "case": case,
        "status": data.get("status"),
        "claim_level": data.get("claim_level"),
        "simulated": data.get("simulated"),
        "path": str(path),
    }
    rows.append(row)
    dma_deferred = case == "30_non_ascon_dma" and data.get("status") == "NOT_RUN"
    if data.get("status") != "PASS":
        item = "%s status=%s" % (case, data.get("status"))
        problems.append(item)
        (dma_deferred_problems if dma_deferred else fatal_problems).append(item)
    if not dma_deferred and case != "00_preflight" and data.get("claim_level") != required_claim:
        item = "%s claim_level=%s" % (case, data.get("claim_level"))
        problems.append(item)
        fatal_problems.append(item)
    if not dma_deferred and case != "00_preflight" and data.get("simulated") is not False:
        item = "%s did not prove simulated=false" % case
        problems.append(item)
        fatal_problems.append(item)

preflight_record = json.loads((run_dir / "00_preflight.json").read_text(encoding="utf-8"))
l2_context = preflight_record.get("l2_evidence")
if l2_context is None:
    item = "preflight lacks l2_evidence; rerun l3_00_preflight.ipynb with the updated Notebook"
    problems.append(item)
    fatal_problems.append(item)

allow_partial = os.environ.get("L3_ALLOW_PARTIAL") == "1"  # 仅缺 DMA 时可显式允许 PARTIAL
partial_allowed = allow_partial and bool(dma_deferred_problems) and not fatal_problems
status = "PASS" if not problems else ("PARTIAL" if partial_allowed else "FAIL")
summary = {
    "case": "summary",
    "status": status,
    "claim_level": "hardware_measured" if status == "PASS" else "not_measured",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "run_id": run_dir.name,
    "run_dir": str(run_dir),
    "rows": rows,
    "problems": problems,
    "l2_context": l2_context,  # L2 remains simulated; this field never upgrades it to hardware evidence
    "allow_partial": allow_partial,
    "allowed_claim": (
        "Manual L3 pass for Ascon release loading, non-Ascon AXI GPIO and AXI DMA loopback."
        if status == "PASS"
        else (
            "Partial L3 evidence only; do not claim complete non-Ascon L3 pass."
            if status == "PARTIAL"
            else "L3 failed or required evidence is unavailable; make no L3 pass claim."
        )
    ),
}
write_json(run_dir / "40_summary.json", summary)
lines = [
    "# PYNQ Skill L3 summary",
    "",
    "- run: `%s`" % run_dir,
    "- status: **%s**" % status,
    "- claim: %s" % summary["allowed_claim"],
    "",
    "| Case | Status | Claim | Simulated |",
    "|---|---|---|---|",
]
for row in rows:
    lines.append(
        "| %s | %s | %s | %s |"
        % (
            row["case"],
            row.get("status"),
            row.get("claim_level"),
            row.get("simulated"),
        )
    )
if problems:
    lines.extend(["", "## Problems", ""])
    lines.extend("- " + item for item in problems)
(run_dir / "40_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
if status == "FAIL":
    raise AssertionError(problems)

# PYNQ Skill L3 summary

- run: `/home/xilinx/jupyter_notebooks/amd_fpga_2026/data/skill/l3_20261031_063709`
- status: **PASS**
- claim: Manual L3 pass for Ascon release loading, non-Ascon AXI GPIO and AXI DMA loopback.

| Case | Status | Claim | Simulated |
|---|---|---|---|
| 00_preflight | PASS | environment | None |
| 10_ascon_release | PASS | hardware_measured | False |
| 20_non_ascon_gpio | PASS | hardware_measured | False |
| 30_non_ascon_dma | PASS | hardware_measured | False |
